# 10 — Grouped queries, QK normalization, and honest cost accounting

Grouped-query attention uses Hq query heads but fewer Hkv key/value heads, with Hq divisible by Hkv. Each query head retains its own attention distribution. MHA is Hkv=Hq; MQA is Hkv=1.

This lab's optional QK normalization is learned per-head RMSNorm before RoPE, followed by the usual division by sqrt(d). That is an explicit variant, not a universal definition of “QK norm.”

We finish by assembling a modern tiny decoder and separating stored parameters, logical KV payload, dense arithmetic estimates, and measured runtime.

## How to work through this notebook

Run setup once. At each checkpoint, write a prediction and try the small implementation before reading its adjacent reference solution. All reference cells run unchanged from top to bottom; exercise cells contain safe, optional starting points. Numerical checks use CPU float64 unless explicitly noted. Agent-verified reference execution is separate from your learning progress.

In [ ]:
from pathlib import Path
import sys, copy, math, inspect
from dataclasses import replace
import torch
from torch import nn
from torch.nn import functional as F
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/dongxi_llms/decoder_lab.py").exists()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the Dongxi_LLMs repository")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from dongxi_llms.decoder_lab import (
    DecoderConfig, TinyDecoder, DecoderBlock, MultiHeadAttention, MLP, RMSNorm,
    layer_norm, rms_norm, rope, attend, parameter_count, analytical_parameters,
    cost_estimate, teaching_batch, next_token_loss, fit_one_batch)
torch.set_num_threads(1)
torch.manual_seed(505)
DTYPE = torch.float64
def close(actual, expected, atol=1e-10, rtol=1e-8):
    torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol)
print("CPU reference environment:", torch.__version__)


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
from dongxi_llms import decoder_visuals as viz
def show_visual(figure):
    display(figure)
    plt.close(figure)


## Architecture map — your location in the model

The highlighted stage is this lesson’s focus. B = batch, T = positions, D = model width, V = vocabulary size. This is a structural map, not measured activations or runtime. The modern route uses RoPE inside attention rather than adding learned absolute positions.

![Architecture map — your location in the model. The highlighted stage is this lesson’s focus. B = batch, T = positions, D = model width, V = vocabulary size. This is a structural map, not measured activations or runtime. The modern route uses RoPE inside attention rather than adding learned absolute positions.](../figures/chapter-05/day-06-03_gqa_qknorm_and_costs-architecture-map.png)

*Saved architecture schematic. The following cell regenerates it; it does not execute or train a model.*

In [ ]:
from dongxi_llms import decoder_architecture as architecture
show_visual(architecture.model_map(focus='gqa', modern=True))

## Keep distinct query heads while sharing source projections

For GQA, Hq is a multiple of Hkv. Query heads within a group share K/V source heads, but retain their own distributions. The optional per-head QK normalization used later in the notebook is omitted in this basic GQA close-up; it belongs after projection/head split and before RoPE.

![Keep distinct query heads while sharing source projections. For GQA, Hq is a multiple of Hkv. Query heads within a group share K/V source heads, but retain their own distributions. The optional per-head QK normalization used later in the notebook is omitted in this basic GQA close-up; it belongs after projection/head split and before RoPE.](../figures/chapter-05/day-06-03_gqa_qknorm_and_costs-architecture-detail.png)

*Saved architecture schematic. The following cell regenerates it; it does not execute or train a model.*

Projection matrix shapes use the mathematical row-vector convention `X @ W`; PyTorch `Linear.weight` stores the transpose of this displayed matrix.

This close-up isolates grouped head sharing: RoPE and optional Q/K normalization are omitted. In the modern decoder, Q/K normalization (when enabled) precedes RoPE, and both occur before the attention scores.


In [ ]:
show_visual(architecture.attention_detail(grouped=True))

## 1. Compare grouped K/V with explicit repetition

Compute grouped attention and compare it to an independent scaled-dot-product reference with repeated K/V tensors. Do their gradients agree?

**Your prediction:** _Write it here before running the reference._

In [ ]:
q = torch.randn(2, 4, 6, 4, dtype=DTYPE, requires_grad=True)
k = torch.randn(2, 2, 6, 4, dtype=DTYPE, requires_grad=True)
v = torch.randn(2, 2, 6, 4, dtype=DTYPE, requires_grad=True)
# Your implementation: repeat K/V across the appropriate query-head groups.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
grouped, weights = attend(q, k, v)
expanded_k, expanded_v = k.repeat_interleave(2, 1), v.repeat_interleave(2, 1)
reference = F.scaled_dot_product_attention(q, expanded_k, expanded_v, is_causal=True)
close(grouped, reference)
probe = torch.randn_like(grouped)
a = torch.autograd.grad((grouped*probe).sum(), (q,k,v))
b = torch.autograd.grad((reference*probe).sum(), (q,k,v))
for ga, gb in zip(a,b): close(ga,gb)
print("Compact K / expanded K / weights:", k.shape, expanded_k.shape, weights.shape)
for kv_heads in (1, 4):
    kk = torch.randn(2, kv_heads, 6, 4, dtype=DTYPE)
    vv = torch.randn_like(kk)
    out, _ = attend(q.detach(), kk, vv)
    ref = F.scaled_dot_product_attention(q.detach(), kk.repeat_interleave(4//kv_heads,1),
            vv.repeat_interleave(4//kv_heads,1), is_causal=True)
    close(out, ref)
print("MQA and MHA endpoint checks passed.")

### Why this works

Shared K/V receive summed feedback from the query heads using them. This transparent implementation physically repeats K/V during attention; it demonstrates the mathematics, not an optimized grouped kernel.

### Visual explanation — See which heads share keys and values

Four query heads form two groups. Heads in a group share K/V projections, but their queries—and thus their attention distributions—can differ. Arrows indicate sharing, not attention probabilities.

The figure uses this lesson’s tensors. Rerun it after changing the preceding experiment.

![See which heads share keys and values. Four query heads form two groups. Heads in a group share K/V projections, but their queries—and thus their attention distributions—can differ. Arrows indicate sharing, not attention probabilities.](../figures/chapter-05/day-06-03_gqa_qknorm_and_costs-visual-gqa-routing.png)

*Saved reference preview. The code below regenerates this figure from the current lesson state; it does not overwrite the preview.*

In [ ]:
show_visual(viz.grouped_heads(4,2))

## 2. Isolate QK normalization

Scale a query and key by the same positive constant. Compare raw scores with RMS-normalized scores; include epsilon in your interpretation.

**Your prediction:** _Write it here before running the reference._

In [ ]:
# Predict which score computation is less sensitive to common rescaling.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
query, key = q.detach(), expanded_k.detach()
unit_scale = torch.ones(4, dtype=DTYPE)
raw = query @ key.transpose(-1,-2) / 2
raw_scaled = (10*query) @ (10*key).transpose(-1,-2) / 2
normalized = rms_norm(query, unit_scale) @ rms_norm(key, unit_scale).transpose(-1,-2) / 2
normalized_scaled = rms_norm(10*query, unit_scale) @ rms_norm(10*key, unit_scale).transpose(-1,-2) / 2
close(raw_scaled, 100*raw)
print("Raw score change norm:", float((raw_scaled-raw).norm()))
print("Normalized score change norm:", float((normalized_scaled-normalized).norm()))
assert (normalized_scaled-normalized).norm() < (raw_scaled-raw).norm()
cfg = DecoderConfig(modern=True, kv_heads=2)
plain = TinyDecoder(cfg).double()
qkn = TinyDecoder(replace(cfg, qk_norm=True)).double()
missing, unexpected = qkn.load_state_dict(plain.state_dict(), strict=False)
assert not unexpected and all("q_norm.weight" in n or "k_norm.weight" in n for n in missing)
ids, labels = teaching_batch()
for candidate in (plain, qkn):
    value = next_token_loss(candidate(ids), labels)
    value.backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in candidate.parameters())
print("QK-normalized model loss:", float(value.detach()))

### Why this works

Positive scale invariance is approximate when epsilon is nonzero. Learned RMS scales and the retained sqrt(d) factor affect the score distribution. This is a connectivity and sensitivity check, not a trained stability comparison.

## 3. Account for compact caches and dense arithmetic

Vary only the number of KV heads. Reconcile analytical parameters and cache bytes with actual tensors. Does the number of query attention matrices shrink?

**Your prediction:** _Write it here before running the reference._

In [ ]:
# Your implementation: count parameters and K/V tensor payload bytes.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
ids, _ = teaching_batch()
rows = []
for kv_heads in (4, 2, 1):
    config = DecoderConfig(modern=True, qk_norm=True, kv_heads=kv_heads)
    model = TinyDecoder(config).double().eval()
    with torch.no_grad():
        full, cache = model(ids, return_cache=True)
        _, prefix_cache = model(ids[:, :2], return_cache=True)
        suffix = model(ids[:, 2:], caches=prefix_cache)
        changed = ids.clone(); changed[:, 4:] = 0
        close(model(changed)[:, :4], full[:, :4])
    close(suffix, full[:, 2:])
    estimate = cost_estimate(config, batch=2, length=6, bytes_per_element=8)
    payload = sum(t.numel()*t.element_size() for pair in cache for t in pair)
    assert payload == estimate["logical_kv_bytes"]
    assert parameter_count(model) == estimate["parameters"]
    rows.append({"kv_heads": kv_heads, **estimate})
print(*rows, sep="\n")
assert rows[0]["logical_kv_bytes"] == 2*rows[1]["logical_kv_bytes"] == 4*rows[2]["logical_kv_bytes"]

### Why this works

Logical KV bytes = 2 × layers × batch × Hkv × length × d × bytes/element. Hq attention distributions remain. The FLOP estimate counts dense forward matrix multiplications with multiply-add=2, including all-position vocabulary projection; it excludes softmax, norms, activations, backward, and allocator overhead. No latency speedup is measured.

### Visual explanation — Compare storage, cache, and arithmetic separately

These are the measured counts and analytical estimates from the preceding cell. Cache bars count compact tensor payloads; FLOPs do not measure real latency.

The figure uses this lesson’s tensors. Rerun it after changing the preceding experiment.

![Compare storage, cache, and arithmetic separately. These are the measured counts and analytical estimates from the preceding cell. Cache bars count compact tensor payloads; FLOPs do not measure real latency.](../figures/chapter-05/day-06-03_gqa_qknorm_and_costs-visual-gqa-cost.png)

*Saved reference preview. The code below regenerates this figure from the current lesson state; it does not overwrite the preview.*

In [ ]:
show_visual(viz.costs(rows))

## 4. Map the toy design to a pinned real configuration

Inspect a small frozen set of Qwen3 configuration fields without downloading weights. Must model width equal query-head count times head width?

**Your prediction:** _Write it here before running the reference._

In [ ]:
# Predict before reading the pinned configuration fields.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
qwen = dict(hidden_size=1024, num_attention_heads=16, num_key_value_heads=8,
            head_dim=128, intermediate_size=3072, num_hidden_layers=28,
            vocab_size=151936, tie_word_embeddings=True, rope_theta=1000000)
print(qwen)
print("Residual width:", qwen["hidden_size"])
print("Concatenated query-head width:", qwen["num_attention_heads"]*qwen["head_dim"])
print("Q projection [out,in]:", (16*128, 1024))
print("Attention output projection [out,in]:", (1024, 16*128))
assert qwen["hidden_size"] != qwen["num_attention_heads"]*qwen["head_dim"]
# Accounting-only candidates: do NOT instantiate or train these larger designs.
for width, layers, hidden in [(512,15,1408), (640,21,1728), (768,23,2048)]:
    candidate = DecoderConfig(vocab=16000, width=width, heads=width//64,
        kv_heads=2, head_dim=64, layers=layers, hidden=hidden,
        modern=True, qk_norm=True, max_length=2048)
    print("Candidate (D,L,FFN):", (width,layers,hidden),
          "parameters:", analytical_parameters(candidate))

### Why this works

The residual width and concatenated head width can differ because learned projections connect them. Source: [Qwen3-0.6B config at c1899de](https://huggingface.co/Qwen/Qwen3-0.6B/raw/c1899de289a04d12100db370d81485cdf75e47ca/config.json), checked 2026-09-06. The toy decoder is not checkpoint compatible: its RoPE layout/base and other choices differ. The three larger configurations are approximate scale-design exercises, not allocated or benchmarked models.

## Takeaway and evidence boundary

Architecture defense: trace one token through the modern model, identify every shape and gradient path, and explain which costs GQA reduces. Optional next session: reuse a block without confusing stored weights with computation.

Companion map: [Chapter 5 pathway](../day-05/README.md). Reusable source: [decoder_lab.py](../../src/dongxi_llms/decoder_lab.py). Record your explanation and remaining questions here; the notebook's existence does not mark the lesson complete.